# II. Per too analysis

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as w
from IPython.display import display, HTML
from functools import lru_cache

from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "helper.py").exists()), Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
import helper as st

st.setup_plotly()
TOOLS = st.load_tools()
HOLES = st.load_holes()
PROGRAM = st.load_program()
STEP_RESULTS = st.load_step_results().merge(HOLES[["file_id", "hole_idx", "cycle_ok"]], on="file_id")
STEP_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # step 0..3, fixed whatever the tool


@lru_cache(maxsize=4)
def tool_data(tool_ref):
    # Everything the sections need for one tool, computed once per tool (~0.5 s)
    holes = HOLES[HOLES.tool_ref == tool_ref].reset_index(drop=True)
    series = st.load_series([tool_ref])
    valid = st.valid_files(holes, series)
    series = series[series.file_id.isin(valid)].merge(holes[["file_id", "hole_idx"]], on="file_id")
    info = holes[["file_id", "hole_idx", "cycle_ok"]]
    return dict(
        holes=holes,
        valid=valid,
        series=series,
        profiles=st.depth_profiles(series).merge(info, on="file_id"),
        steps=st.hole_step_features(series, holes, PROGRAM),
        program=PROGRAM[PROGRAM.tool_ref == tool_ref],
    )

def add_step_bands(fig, d, row=None):
    # Shade the median depth range of each cutting step
    spans = d["steps"].groupby("step")[["depth_start", "depth_end"]].median()
    for i, (step, r) in enumerate(spans.iterrows()):
        fig.add_vrect(x0=r.depth_start, x1=r.depth_end, fillcolor=st.MUTED, opacity=0.03 if i % 2 else 0.07,
                      line_width=0, layer="below", row=row, col=1 if row else None)
        fig.add_annotation(x=(r.depth_start + r.depth_end) / 2, y=1.0, yref="paper", yanchor="bottom",
                           text=st.step_label(step), showarrow=False, font=dict(size=11, color=st.INK_2))

def step_title(d, step):
    # "Step 2 · 1600 rpm · rapide", using the most frequent set-point of that step
    p = d["program"][d["program"].step == step]
    if p.empty:
        return st.step_label(step)
    return f"{st.step_label(step)} · {p.RPM.mode()[0]:.0f} rpm · {p.regime.mode()[0]}"


def hole_colorbar(fig, n):
    # Invisible trace that only carries the colour scale of the hole order
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",
                             marker=dict(colorscale=st.SEQ_BLUES, cmin=1, cmax=n, color=[1], showscale=True,
                                         colorbar=dict(title="N° trou", thickness=12, len=0.8))))

## A. Tool Selection

Select the tool to run the analysis on

In [ ]:
tool_dd = w.Dropdown(options=list(zip(TOOLS.label, TOOLS.tool_ref)), description="Outil",
                     layout=w.Layout(width="360px"))
D = lambda: tool_data(tool_dd.value)
display(tool_dd)

## B. Tool Card

Overview of the tool statistics, etc.  
**Note** : 
- Feed (mm/s) = Feed (mm/tr) × RPM / 60, represent how far is goes per turn or second in the item.
- Gap here represent the pullback to free a bit of materials (debourrage).

In [ ]:
def render_card():
    d = D(); h = d["holes"]
    duration = h.date.max() - h.date.min()
    display(HTML(st.tiles([
        ("Trous", len(h)),
        ("Cycles OK", f"{h.cycle_ok.sum()} ({h.cycle_ok.mean():.0%})"),
        ("Période", f"{h.date.min():%d/%m %H:%M} → {h.date.max():%d/%m %H:%M}"),
        ("Durée totale", f"{duration.total_seconds() / 3600:.1f} h"),
        ("Temps de cycle médian", f"{h.cycle_time_s.median():.0f} s"),
        ("Profondeur médiane", f"{h.distance_mm.median():.1f} mm"),
        ("Tête(s)", ", ".join(h.head_name.unique())),
    ])))

    # Program set-points: one row per step, "a / b" when they changed between holes
    cols = ["RPM", "Feed (mm/tr)", "Feed (mm/s)", "Torque Min (A)", "Torque Max (A)", "Gap (mm)", "regime"]
    fmt = lambda s: " / ".join(f"{v:g}" if isinstance(v, float) else str(v) for v in sorted(s.unique()))
    prog = d["program"].groupby("step")[cols].agg(fmt)
    prog.insert(0, "trous", d["program"].groupby("step").size())
    prog.index = prog.index.map(st.step_label)
    display(HTML("<b>Programme (valeurs distinctes rencontrées)</b>"), prog)

    sr = STEP_RESULTS[STEP_RESULTS.tool_ref == tool_dd.value]
    codes = pd.crosstab(sr.step.map(st.step_label), sr["Stop Code"].astype(int), margins=True, margins_name="Total")
    display(HTML("<b>Codes d'arrêt par step.</b>"), codes)

st.live(render_card, watch=[tool_dd]);

## C. Drilling Curves

Each curve is one drilling application. The slide **UP** looking at the torque for instance signals the wear (more torque is needed as the tool wears down). 

Note that each curve, when looking by depth, is sampled each 0.1mm to avoid computing the pullback (if pullback there is, then the value is ignored)


In [ ]:
sig_dd = w.Dropdown(options=st.SIGNALS, value="I Torque (A)", description="Signal")
x_tb = w.ToggleButtons(options=["Profondeur", "Temps"], description="Axe X", style={"button_width": "90px"})
rng = w.IntRangeSlider(description="Trous", min=1, max=2, value=(1, 2), continuous_update=False,
                       layout=w.Layout(width="420px"))
stride = w.IntSlider(description="1 trou sur", min=1, max=10, value=1, continuous_update=False)
color_tb = w.ToggleButtons(options=["Ordre du trou", "OK / KO"], description="Couleur", style={"button_width": "110px"})


def reset_range(*_):
    n = len(D()["holes"])
    rng.max = n
    rng.value = (1, n)

tool_dd.observe(reset_range, "value"); reset_range()


def render_curves():
    d = D(); n = len(d["holes"]); col = sig_dd.value
    sel = d["holes"][d["holes"].hole_idx.between(*rng.value)].iloc[::stride.value]
    sel = sel[sel.file_id.isin(d["valid"])]
    if x_tb.value == "Profondeur":
        data, xcol, xlab = d["profiles"], "depth_bin", "Profondeur (mm)"
    else:
        data, xcol, xlab = d["series"].iloc[::5], "t_s", "Temps depuis le début du cycle (s)"
    curves = dict(tuple(data[data.file_id.isin(sel.file_id)].groupby("file_id")))

    fig = go.Figure()
    for h in sel.itertuples():
        c = curves.get(h.file_id)
        if c is None:
            continue
        if color_tb.value == "Ordre du trou":
            color, width = st.seq_color((h.hole_idx - 1) / max(n - 1, 1)), 1
        else:
            color, width = (st.MUTED, 0.8) if h.cycle_ok else (st.KO_COLOR, 1.6)
        status = "" if h.cycle_ok else " · KO"
        fig.add_trace(go.Scattergl(x=c[xcol], y=c[col], mode="lines", line=dict(color=color, width=width),
                                   showlegend=False,
                                   hovertemplate=f"Trou {h.hole_idx}{status}<br>x=%{{x:.2f}}<br>y=%{{y:.2f}}<extra></extra>"))
    if color_tb.value == "Ordre du trou":
        hole_colorbar(fig, n)
    else:
        for name, color in [("Cycle OK", st.MUTED), ("Cycle KO", st.KO_COLOR)]:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color=color, width=2), name=name))
    if x_tb.value == "Profondeur":
        add_step_bands(fig, d)
    fig.update_layout(title=f"{col} — {tool_dd.label} — {len(sel)} trous", xaxis_title=xlab, yaxis_title=col,
                      height=520, hovermode="closest")
    fig.show()

st.live(render_curves, [sig_dd, x_tb, rng, stride, color_tb], watch=[tool_dd]);

## E. Depth Profiles

Each line is a whole from top -> first / bottom -> last, the color maps the value.  
The point of this graph is to have a visual understanding of the evolution of some of the values as the tool drills deeper into the material over time.
-   **Ex:** Does wear cause overtime the drill to use more and more torque per drilling ? do we have consistency ? etc.

In [ ]:
heat_sig = w.Dropdown(options=st.SIGNALS, value="I Torque (A)", description="Signal")

def render_heatmap():
    d = D(); col = heat_sig.value
    pv = d["profiles"].pivot_table(index="hole_idx", columns="depth_bin", values=col)
    lo, hi = np.nanpercentile(pv.values, [1, 99])
    fig = go.Figure(go.Heatmap(
        z=pv.values, x=pv.columns, y=pv.index, zmin=lo, zmax=hi,
        colorscale=["#f4f8fd", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"],
        colorbar=dict(title=col, thickness=12),
        hovertemplate="Trou %{y}<br>%{x:.1f} mm<br>%{z:.2f}<extra></extra>"))
    ko = d["holes"][~d["holes"].cycle_ok & d["holes"].file_id.isin(d["valid"])]
    fig.add_trace(go.Scatter(x=[pv.columns.min() - 0.6] * len(ko), y=ko.hole_idx, mode="markers", name="Cycle KO",
                             marker=dict(symbol="x", color=st.KO_COLOR, size=7)))
    fig.update_yaxes(autorange="reversed", title="N° de trou")
    fig.update_layout(title=f"{col} par trou et profondeur — {tool_dd.label}", xaxis_title="Profondeur (mm)",
                      height=560, legend=dict(orientation="h", y=1.06, x=1, xanchor="right"))
    fig.show()

st.live(render_heatmap, [heat_sig], watch=[tool_dd]);

# F. Distribution profile as wear stacks up over time

We display the average of the 5 top / bottom drills, the median of all the holes made by the drill and the 10-90% band. This enables us to look at were we have to most variance / volatility in the behavior of a tool and at which depth.

In [ ]:
prof_sig = w.Dropdown(options=st.SIGNALS, value="I Torque (A)", description="Signal")
n_first = w.IntSlider(description="N trous", min=2, max=20, value=5, continuous_update=False)
ok_only = w.Checkbox(value=True, description="Cycles OK uniquement")

def render_profile():
    d = D(); col = prof_sig.value; N = n_first.value
    p = d["profiles"][d["profiles"].cycle_ok] if ok_only.value else d["profiles"]
    q = p.groupby("depth_bin")[col].quantile([0.1, 0.5, 0.9]).unstack()
    q = q[p.groupby("depth_bin").size() >= 3]
    idx = sorted(p.hole_idx.unique())
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=q.index, y=q[0.9], mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=q.index, y=q[0.1], mode="lines", line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(137,135,129,0.22)", name="Bande 10–90 %", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=q.index, y=q[0.5], mode="lines", line=dict(color=st.INK_2, width=1.5), name="Médiane"))
    for name, holes_, color in [(f"{N} premiers trous", idx[:N], st.SEQ_BLUES[0]),
                                (f"{N} derniers trous", idx[-N:], st.SEQ_BLUES[-1])]:
        m = p[p.hole_idx.isin(holes_)].groupby("depth_bin")[col].mean()
        fig.add_trace(go.Scatter(x=m.index, y=m.values, mode="lines", line=dict(color=color, width=2), name=name))
    add_step_bands(fig, d)
    fig.update_layout(title=f"Profil type de {col} — {tool_dd.label}", xaxis_title="Profondeur (mm)",
                      yaxis_title=col, height=480, hovermode="x unified")
    fig.show()

st.live(render_profile, [prof_sig, n_first, ok_only], watch=[tool_dd]);

## G. Hole per Hole Wear Evolution

This cell follows the same hole plotting each variable looking at their evolution as more holes are drilled.

We also look at the change in percentage between the 5 first and 5 last holes.

In [ ]:
kpi_dd = w.Dropdown(options={v: k for k, v in st.KPIS.items()}, value="torque_mean", description="Indicateur")
win = w.IntSlider(description="Fenêtre", min=1, max=21, step=2, value=7, continuous_update=False)
with_approach = w.Checkbox(value=False, description="Inclure l'approche (step 0)")

def render_evolution():
    d = D(); kpi = kpi_dd.value
    f = d["steps"] if with_approach.value else d["steps"][d["steps"].regime != "approche"]
    steps = sorted(f.step.unique())
    titles = []
    for s in steps:
        g = f[(f.step == s) & f.cycle_ok]
        _, rel = st.trend(g.hole_idx, g[kpi])
        titles.append(f"{step_title(d, s)} — début→fin {rel:+.0%}" if np.isfinite(rel) else step_title(d, s))
    fig = make_subplots(rows=len(steps), cols=1, shared_xaxes=True, vertical_spacing=0.08, subplot_titles=titles)
    for i, s in enumerate(steps, 1):
        g = f[f.step == s].sort_values("hole_idx")
        ok, ko = g[g.cycle_ok], g[~g.cycle_ok]
        fig.add_trace(go.Scatter(x=ok.hole_idx, y=ok[kpi], mode="markers", name="Cycle OK", legendgroup="ok",
                                 showlegend=i == 1, marker=dict(color=st.SEQ_BLUES[4], size=7, opacity=0.8)), row=i, col=1)
        fig.add_trace(go.Scatter(x=ko.hole_idx, y=ko[kpi], mode="markers", name="Cycle KO", legendgroup="ko",
                                 showlegend=i == 1, marker=dict(color=st.KO_COLOR, size=8, symbol="x")), row=i, col=1)
        roll = ok[kpi].rolling(win.value, center=True, min_periods=1).median()
        fig.add_trace(go.Scatter(x=ok.hole_idx, y=roll, mode="lines", name="Médiane glissante (OK)", legendgroup="roll",
                                 showlegend=i == 1, line=dict(color=st.INK_2, width=2)), row=i, col=1)
        fig.update_yaxes(title_text=st.KPIS[kpi], row=i, col=1)
    fig.update_xaxes(title_text="N° de trou", row=len(steps), col=1)
    fig.update_layout(height=250 * len(steps) + 90, title=f"{st.KPIS[kpi]} au fil des trous — {tool_dd.label}",
                      margin=dict(t=110), legend=dict(orientation="h", yanchor="bottom", y=1.0, yref="container", x=1, xanchor="right"))
    fig.show()

    # Variation début -> fin of every KPI, per step (OK cycles)
    rows = {}
    for s in steps:
        g = f[(f.step == s) & f.cycle_ok]
        rows[step_title(d, s)] = {lab: st.trend(g.hole_idx, g[k])[1] for k, lab in st.KPIS.items()}
    table = pd.DataFrame(rows).T
    display(HTML("<b>Variation 10 % premiers → 10 % derniers trous (cycles OK)</b>"),
            table.style.format("{:+.0%}", na_rep="–").background_gradient(cmap="RdBu_r", vmin=-0.5, vmax=0.5, axis=None))

st.live(render_evolution, [kpi_dd, win, with_approach], watch=[tool_dd]);

## H. Focus on one hole

Plotting and display of the significant variables for each hole, with the stop codes overlayed when occuring.


In [ ]:
hole_dd = w.Dropdown(description="Trou", layout=w.Layout(width="360px"))

def reset_holes(*_):
    h = D()["holes"]; h = h[h.file_id.isin(D()["valid"])]
    hole_dd.options = [(f"#{r.hole_idx} · {r.date:%d/%m %H:%M} · {'OK' if r.cycle_ok else 'KO'}", int(r.file_id))
                       for r in h.itertuples()]
    hole_dd.value = hole_dd.options[0][1]  # ipywidgets 8 leaves the value empty when options change

tool_dd.observe(reset_holes, "value"); reset_holes()

ZOOM_ROWS = [("depth_mm", "Profondeur (mm)"), ("I Torque (A)", "Couple (A)"), ("I Thrust (A)", "Poussée (A)"),
             ("Torque Power (W)", "Puissance (W)"), ("Gap Length (mm)", "Gap (mm)"), ("Mem Torque min (A)", "Mem Torque min (A)")]

def render_zoom():
    d = D()
    if hole_dd.value is None:
        return
    c = d["series"][d["series"].file_id == hole_dd.value]
    fig = make_subplots(rows=len(ZOOM_ROWS), cols=1, shared_xaxes=True, vertical_spacing=0.025)
    for i, (col, lab) in enumerate(ZOOM_ROWS, 1):
        fig.add_trace(go.Scattergl(x=c.t_s, y=c[col], mode="lines", line=dict(color=st.SEQ_BLUES[4], width=1.2),
                                   name=lab, showlegend=False), row=i, col=1)
        fig.update_yaxes(title_text=lab, row=i, col=1, title_font=dict(size=11))
    fig.update_yaxes(autorange="reversed", row=1, col=1)
    spans = c.groupby("step").t_s.agg(["min", "max"])
    for j, (s, r) in enumerate(spans.iterrows()):
        fig.add_vrect(x0=r["min"], x1=r["max"], fillcolor=st.MUTED, opacity=0.03 if j % 2 else 0.07, line_width=0,
                      layer="below", row="all", col=1)
        fig.add_annotation(x=(r["min"] + r["max"]) / 2, y=1.0, yref="paper", yanchor="bottom", showarrow=False,
                           text=st.step_label(s), font=dict(size=11, color=st.INK_2))
    stops = c[c["Stop code"] != 0]
    fig.add_trace(go.Scatter(x=stops.t_s, y=stops.depth_mm, mode="markers", name="Code d'arrêt",
                             marker=dict(symbol="x", size=9, color=st.INK), customdata=stops["Stop code"],
                             hovertemplate="code %{customdata:.0f}<br>t=%{x:.2f} s<extra></extra>"), row=1, col=1)
    fig.update_xaxes(title_text="Temps (s)", row=len(ZOOM_ROWS), col=1)
    fig.update_layout(title=f"{tool_dd.label} — trou {hole_dd.label}", height=900, hovermode="x unified",
                      legend=dict(orientation="h", y=1.04, x=1, xanchor="right"))
    fig.show()

    prog = d["program"][d["program"].file_id == hole_dd.value].set_index("step")[
        ["RPM", "Feed (mm/tr)", "Feed (mm/s)", "Torque Min (A)", "Torque Max (A)", "Gap (mm)", "regime"]]
    sr = STEP_RESULTS[STEP_RESULTS.file_id == hole_dd.value].set_index("step")[
        ["Stop Code", "Duration (s)", "Distance M2", "M1 Max Amp", "M1 No Load Amp", "M2 Max Amp", "Gap Max (mm)"]]
    display(HTML("<b>Programme</b>"), prog, HTML("<b>Step results</b>"), sr)

st.live(render_zoom, [hole_dd]);

## I. KO Cycles

Zoom on the KO cycles with their stop codes per steps. They are plotted against the 90 / 10 % Band for comparaison with the average behavior.

In [ ]:
ko_sig = w.Dropdown(options=st.SIGNALS, value="I Torque (A)", description="Signal")

def render_ko():
    d = D(); col = ko_sig.value; h = d["holes"]
    ko = h[~h.cycle_ok]
    if ko.empty:
        display(HTML("<i>Aucun cycle KO pour cet outil.</i>"))
        return
    codes = (STEP_RESULTS[STEP_RESULTS.file_id.isin(ko.file_id)]
             .pivot_table(index="file_id", columns="step", values="Stop Code", aggfunc="first")
             .rename(columns=lambda s: f"code {st.step_label(s)}"))
    table = ko.set_index("file_id")[["hole_idx", "date", "cycle_time_s", "distance_mm"]].join(codes)
    display(HTML(f"<b>{len(ko)} cycles KO sur {len(h)}</b>"), table.set_index("hole_idx"))

    p = d["profiles"]
    q = p[p.cycle_ok].groupby("depth_bin")[col].quantile([0.1, 0.5, 0.9]).unstack()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=q.index, y=q[0.9], mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=q.index, y=q[0.1], mode="lines", line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(137,135,129,0.22)", name="Cycles OK (10–90 %)", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=q.index, y=q[0.5], mode="lines", line=dict(color=st.INK_2, width=1.5), name="Médiane OK"))
    for i, (fid, g) in enumerate(p[p.file_id.isin(ko.file_id)].groupby("file_id")):
        idx = g.hole_idx.iloc[0]
        fig.add_trace(go.Scatter(x=g.depth_bin, y=g[col], mode="lines", line=dict(color=st.KO_COLOR, width=1.2),
                                 name="Cycle KO", legendgroup="ko", showlegend=i == 0,
                                 hovertemplate=f"Trou {idx} (KO)<br>%{{x:.1f}} mm → %{{y:.2f}}<extra></extra>"))
    add_step_bands(fig, d)
    fig.update_layout(title=f"Cycles KO vs cycles OK — {col} — {tool_dd.label}", xaxis_title="Profondeur (mm)",
                      yaxis_title=col, height=460)
    fig.show()

st.live(render_ko, [ko_sig], watch=[tool_dd]);

## J. Correlation matrix of the features

Using Pearson's correlation we look at how each value moves together.  
We added the hole number as a feature to interpret it as the "wear factor".

In [ ]:
corr_step = w.Dropdown(description="Step")

def reset_steps(*_):
    d = D(); steps = sorted(d["steps"][d["steps"].regime != "approche"].step.unique())
    corr_step.options = [(step_title(d, s), int(s)) for s in steps]
    corr_step.value = corr_step.options[0][1]

tool_dd.observe(reset_steps, "value"); reset_steps()

def render_corr():
    d = D()
    if corr_step.value is None:
        return
    g = d["steps"][(d["steps"].step == corr_step.value) & d["steps"].cycle_ok]
    cols = {"hole_idx": "N° de trou"} | {k: v for k, v in st.KPIS.items() if k != "stroke_mm"}
    c = g[list(cols)].rename(columns=cols).corr()
    fig = go.Figure(go.Heatmap(z=c.values, x=c.columns, y=c.index, zmin=-1, zmax=1, colorscale=st.DIVERGING,
                               text=np.round(c.values, 2), texttemplate="%{text}", textfont=dict(size=10),
                               colorbar=dict(title="r", thickness=12)))
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(title=f"Corrélations — {corr_step.label} — {tool_dd.label} ({len(g)} trous OK)", height=620)
    fig.show()

st.live(render_corr, [corr_step], watch=[tool_dd]);